In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
while not (repo_root / "src").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
for candidate in (repo_root, repo_root / "src"):
    candidate_str = str(candidate)
    if candidate.exists() and candidate_str not in sys.path:
        sys.path.insert(0, candidate_str)


In [ ]:
from pathlib import Path
import csv

from src.drive_service.logging_utils import setup_logging
from src.pipeline_paths import build_pipelines_paths
from src.extract_events_from_days_raw import extract_events_from_days_dir


In [ ]:
root = "1FUosjKncLt18JzojmX8tKQm1nbgPI133"
paths = build_pipelines_paths(root)

days_name = "*.days.csv"
days_files = sorted(Path(paths.parsing_output).rglob(days_name))
if not days_files:
    raise FileNotFoundError(
        f"No days files found in {paths.parsing_output} with pattern {days_name}"
    )

paths.parsing_output, paths.events_output, len(days_files), days_files[:5]


In [ ]:
verbose = True
out_name = "events_from_days_raw.csv"
report_json = paths.events_output / "extract_events_from_days_raw.report.json"

max_pattern_examples = 12
max_unmatched_examples_per_file = 5

setup_logging(verbose)

report = extract_events_from_days_dir(
    input_dir=str(paths.parsing_output),
    output_dir=str(paths.events_output),
    days_name=days_name,
    out_name=out_name,
    report_json=str(report_json),
    max_pattern_examples=max_pattern_examples,
    max_unmatched_examples_per_file=max_unmatched_examples_per_file,
)

report["stats"]


In [ ]:
events_pattern = f"*.{out_name}"
events_files = sorted(Path(paths.events_output).rglob(events_pattern))
len(events_files), events_files[:5]


In [ ]:
if events_files:
    sample_events = events_files[0]
    with open(sample_events, "r", encoding="utf-8", newline="") as handle:
        reader = csv.reader(handle)
        sample_rows = []
        for i, row in enumerate(reader):
            sample_rows.append(row)
            if i >= 10:
                break
    sample_events, sample_rows
else:
    "No events CSV files generated"


In [ ]:
files_with_unmatched_rows = report.get("files_with_unmatched_rows", [])
file_errors = report.get("file_errors", [])

{
    "report_json": str(report_json),
    "files_with_unmatched_rows_count": len(files_with_unmatched_rows),
    "file_errors_count": len(file_errors),
    "files_with_unmatched_rows_preview": files_with_unmatched_rows[:3],
    "file_errors_preview": file_errors[:3],
}
